In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG   = "clutchlytics"
SCHEMA    = "bronze"
VOLUME    = "nhl_raw"
TABLE     = f"{CATALOG}.{SCHEMA}.raw_nhl_rosters"
 
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
 
print(f"Source : {VOLUME_PATH}")
print(f"Target : {TABLE}")

In [0]:
# ── DISCOVER ROSTER FILES ────────────────────────────────────────────────────
# Filter Volume contents to only roster files.
# Expected naming: {team_id}_{abbreviation}_roster.json
 
all_files    = dbutils.fs.ls(VOLUME_PATH)
roster_files = [f for f in all_files if f.name.endswith("_roster.json")]
 
print(f"Roster files found: {len(roster_files)}")
for f in sorted(roster_files, key=lambda x: x.name):
    print(f"  {f.name:<35} {f.size:>8,} bytes")

In [0]:
# ── PARSE + FLATTEN ALL ROSTERS ──────────────────────────────────────────────
 
import json
from datetime import datetime, timezone
 
ingested_at = datetime.now(timezone.utc).isoformat()
rows        = []
skipped     = []
 
for f in roster_files:
    filename = f.name  # e.g. "1_bos_roster.json"
 
    # ── Parse team_id and abbreviation from filename ──
    # Format: {team_id}_{abbreviation}_roster.json
    try:
        parts       = filename.replace("_roster.json", "").split("_")
        team_id     = parts[0]
        abbreviation = parts[1].upper()
    except IndexError:
        skipped.append(filename)
        print(f"  SKIPPED (unexpected filename format): {filename}")
        continue
 
    # ── Read and parse JSON ──
    try:
        raw_text  = spark.read.text(f.path)
        json_str  = "\n".join([row.value for row in raw_text.collect()])
        data      = json.loads(json_str)
    except Exception as e:
        skipped.append(filename)
        print(f"  SKIPPED (parse error) {filename}: {e}")
        continue
 
    # ── Navigate ESPN roster structure ──
    # { "data": { "athletes": [ { "position": "...", "items": [ {...athlete...} ] } ] } }
    # Note: Extra "data" wrapper (same as teams.json)
    athlete_groups = data.get("data", {}).get("athletes", [])
 
    team_players = 0
    for group in athlete_groups:
        position_group = group.get("position", "Unknown")
        athletes       = group.get("items", [])
 
        for athlete in athletes:
            rows.append({
                # ── Team context (from filename) ──
                "team_id":              team_id,
                "team_abbreviation":    abbreviation,
 
                # ── Athlete identity ──
                "athlete_id":           athlete.get("id"),
                "uid":                  athlete.get("uid"),
                "full_name":            athlete.get("fullName"),
                "display_name":         athlete.get("displayName"),
                "short_name":           athlete.get("shortName"),
                "first_name":           athlete.get("firstName"),
                "last_name":            athlete.get("lastName"),
 
                # ── Position ──
                "position_group":       position_group,
                "position_name":        athlete.get("position", {}).get("name"),
                "position_abbr":        athlete.get("position", {}).get("abbreviation"),
 
                # ── Jersey + status ──
                "jersey":               athlete.get("jersey"),
                "status":               athlete.get("status", {}).get("name"),
                "status_abbr":          athlete.get("status", {}).get("abbreviation"),
 
                # ── Physical ──
                "age":                  athlete.get("age"),
                "date_of_birth":        athlete.get("dateOfBirth"),
                "birth_city":           athlete.get("birthPlace", {}).get("city"),
                "birth_state":          athlete.get("birthPlace", {}).get("state"),
                "birth_country":        athlete.get("birthPlace", {}).get("country"),
                "height":               athlete.get("height"),
                "weight":               athlete.get("weight"),
                "experience_years":     athlete.get("experience", {}).get("years"),
 
                # ── Shoots / catches ──
                "shoots_catches":       athlete.get("hand", {}).get("abbreviation"),
 
                # ── Ingestion metadata ──
                "season":               2026,
                "source_file":          filename,
                "ingested_at":          ingested_at,
            })
            team_players += 1
 
    print(f"  {filename:<35} → {team_players:>2} players")
 
print(f"\nTotal rows built : {len(rows)}")
print(f"Files skipped    : {len(skipped)}")

In [0]:
# ── WRITE TO DELTA ────────────────────────────────────────────────────────────────────────────────────────────────────
# Overwrite on each run — full refresh from source files.
 
if not rows:
    raise ValueError("No rows to write — check file parsing above before proceeding.")

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

# Define explicit schema to handle nullable fields
schema = StructType([
    StructField("team_id", StringType(), False),
    StructField("team_abbreviation", StringType(), False),
    StructField("athlete_id", StringType(), True),
    StructField("uid", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("display_name", StringType(), True),
    StructField("short_name", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("position_group", StringType(), True),
    StructField("position_name", StringType(), True),
    StructField("position_abbr", StringType(), True),
    StructField("jersey", StringType(), True),
    StructField("status", StringType(), True),
    StructField("status_abbr", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("birth_city", StringType(), True),
    StructField("birth_state", StringType(), True),
    StructField("birth_country", StringType(), True),
    StructField("height", FloatType(), True),
    StructField("weight", FloatType(), True),
    StructField("experience_years", IntegerType(), True),
    StructField("shoots_catches", StringType(), True),
    StructField("season", IntegerType(), False),
    StructField("source_file", StringType(), False),
    StructField("ingested_at", StringType(), False),
])
 
rosters_df = spark.createDataFrame(rows, schema=schema)
 
(
    rosters_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE)
)
 
print(f"Written to {TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
result = spark.sql(f"""
    SELECT
        team_id,
        team_abbreviation,
        COUNT(*)            AS player_count,
        COUNT(CASE WHEN position_abbr = 'G'  THEN 1 END) AS goalies,
        COUNT(CASE WHEN position_abbr = 'D'  THEN 1 END) AS defensemen,
        COUNT(CASE WHEN position_abbr != 'G'
                    AND position_abbr != 'D' THEN 1 END) AS forwards
    FROM {TABLE}
    GROUP BY team_id, team_abbreviation
    ORDER BY team_abbreviation
""")
 
print(f"Teams loaded: {result.count()}")
result.show(32, truncate=False)

In [0]:
# ── QUICK SANITY CHECKS ───────────────────────────────────────────────────────
 
total = spark.sql(f"SELECT COUNT(*) AS total FROM {TABLE}").collect()[0]["total"]
nulls = spark.sql(f"SELECT COUNT(*) AS nulls FROM {TABLE} WHERE athlete_id IS NULL").collect()[0]["nulls"]
dupes = spark.sql(f"""
    SELECT COUNT(*) AS dupes FROM (
        SELECT athlete_id, COUNT(*) AS n
        FROM {TABLE}
        GROUP BY athlete_id
        HAVING n > 1
    )
""").collect()[0]["dupes"]
 
print(f"Total players    : {total}")
print(f"Null athlete_ids : {nulls}  {'<-- investigate' if nulls > 0 else '✓'}")
print(f"Duplicate ids    : {dupes}  {'<-- investigate' if dupes > 0 else '✓'}")
 
# Players on two teams (traded) will show as duplicates -- that is expected.
# Investigate any athlete_id = NULL rows before proceeding to gamelog pulls.